# Model Inference

Load a checkpoint from the RCP and run text continuation. The checkpoint embeds the full training config, so no separate YAML is needed.

In [ ]:
import sys
sys.path.insert(0, "..")

import torch
from pathlib import Path
from transformers import AutoTokenizer
from src.model import build_llama_model

In [ ]:
# ── Edit this path to point at a checkpoint on the RCP ──────────────────────
CHECKPOINT_PATH = Path("../checkpoints/policy_decay_baseline_7k_.../checkpoint_step_7000.pt")
# ────────────────────────────────────────────────────────────────────────────

DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {DEVICE}")

In [ ]:
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
config = ckpt["config"]
print(f"Checkpoint step : {ckpt['step']}")
print(f"Model           : {config['model']['name']}")
print(f"Run name        : {config.get('run_name', '?')}")

tokenizer = AutoTokenizer.from_pretrained(
    config["dataset"].get("tokenizer_name", "gpt2")
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = build_llama_model(config["model"], tokenizer)
model.load_state_dict(ckpt["model"])
model.to(DEVICE)
model.eval()
print(f"Parameters      : {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
def generate(
    prompt: str,
    max_new_tokens: int = 100,
    temperature: float = 0.8,
    top_p: float = 0.9,
) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [ ]:
prompts = [
    "The history of machine learning begins",
    "In order to train a large language model, you need",
    "The capital of France is Paris, and the capital of Germany is",
    "Scientists recently discovered that",
]

for prompt in prompts:
    continuation = generate(prompt)
    print(f"PROMPT:       {prompt}")
    print(f"CONTINUATION: {continuation}")
    print()

## Compare Two Checkpoints (optional)

Load a second checkpoint (e.g. a policy run vs the baseline) and show continuations side-by-side for the same prompt.

In [ ]:
CHECKPOINT_B = Path("../checkpoints/policy_decay_loss_variance_7k_.../checkpoint_step_7000.pt")

ckpt_b = torch.load(CHECKPOINT_B, map_location=DEVICE, weights_only=False)
model_b = build_llama_model(ckpt_b["config"]["model"], tokenizer)
model_b.load_state_dict(ckpt_b["model"])
model_b.to(DEVICE)
model_b.eval()

def generate_b(prompt, **kwargs):
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        output_ids = model_b.generate(
            **inputs, max_new_tokens=80, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output_ids[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)

test_prompt = "The history of machine learning begins"
print(f"PROMPT: {test_prompt}\n")
print(f"[baseline]       {generate(test_prompt, temperature=1.0)}")
print()
print(f"[loss_variance]  {generate_b(test_prompt)}")